# Ciber Assistant — RAG de políticas e procedimentos de segurança
### Versão 3: documentos da Cartilha de Segurança (CERT.br) e da LGPD

Mesma estrutura da versão 2 (configuração → extração → chunking → busca híbrida → contexto enxuto → Claude), com estas mudanças:

- **Leitura com PyMuPDF** e correção dos acentos (a Cartilha grava cedilha e til como marcas separadas).
- **Um bloco por seção ou artigo**: `Art. 48` da LGPD e `2.3 Phishing` da Cartilha, com referência para a busca direta (`artigo 48`, `seção 2.3`).
- **Prompt em português** voltado a políticas e procedimentos, que cita a seção e não inventa artigos ou prazos.

## 0. Preparação do ambiente

Funciona no Colab ou localmente (a versão para VS Code está na pasta do projeto). Coloque os documentos em `DOCS_DIR`: os PDFs da Cartilha e da LGPD.

In [ ]:
!pip install -q sentence-transformers chromadb pandas numpy pymupdf anthropic langchain-text-splitters

import os
import re
import glob
import math
import hashlib
import getpass
import unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import pymupdf
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
from anthropic import Anthropic

print("Dependências e módulos carregados com sucesso.")

### Configuração

Tudo o que se ajusta está aqui. Os que mais mexem no consumo: `PRESUPUESTO_CONTEXTO`, `TOP_K`, `LEX_MIN`, `CLAUDE_MAX_TOKENS` e `CACHE_MODO`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===================== CONFIGURAÇÃO =====================
DOCS_DIR = '/content/drive/MyDrive/Assistente IA/datos'   # PDFs da Cartilha e da LGPD
CHROMA_DIR = './chroma_pdf_db'
CREAR_EJEMPLOS = False           # True: cria um documento de exemplo se a pasta estiver vazia

EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'   # multilíngue (pt/es/en)

# --- Chunking (texto corrido; seções e artigos viram 1 bloco cada) ---
CHUNK_SIZE = 1000                # caracteres
CHUNK_OVERLAP = 0                # sobreposição repete texto (e tokens); suba só se perder contexto nos cortes

# --- Busca híbrida ---
TOP_K = 4
SIM_MINIMA = 0.40                # similaridade mínima do embedding (0.30 deixava passar 'política de férias')
LEX_MIN = 0.30                   # cobertura mínima das palavras-chave da pergunta (peso por raridade)
SIM_ALTA = 0.80                  # acima disso aceita mesmo com baixa cobertura (paráfrases)
PESO_LEXICO = 0.5                # peso da cobertura de palavras-chave no ranking final
MARGEN_RELATIVO = 0.30           # descarta blocos muito piores que o melhor (score do melhor - 0.30)
POOL_CANDIDATOS = 40             # quantos candidatos o Chroma entrega para a reordenação híbrida

# --- Contexto enxuto ---
PRESUPUESTO_CONTEXTO = 900       # máximo de tokens estimados de contexto enviados ao Claude
ELIMINAR_ARTICULOS = False       # True = telegráfico (remove o/a/um/el/la/the...): economiza um pouco, lê pior

# --- Claude ---
CLAUDE_MODEL = os.getenv('CLAUDE_MODEL', 'claude-haiku-5-5')   # troque pelo modelo disponível na sua conta
CLAUDE_MAX_TOKENS = 1000
TEMPERATURA = None               # ex.: 0 para respostas mais estáveis (alguns modelos novos não aceitam esse parâmetro)
DESACTIVAR_THINKING = True      # True: envia thinking={'type': 'disabled'} (use se a saída vier muito acima do texto visível)

# --- Prompt caching ---
CACHE_MODO = 'auto'              # 'auto' | 'siempre' | 'nunca'
CACHE_MIN_TOKENS = 1200          # mínimo estimado para o 'auto' (a API exige ~1.024+, conforme o modelo)

CHARS_POR_TOKEN = 2.5            # será calibrado com a API na Tarefa 2

def tokens_aprox(texto):
    return max(1, int(len(texto) / CHARS_POR_TOKEN))

print(f"Embeddings: {EMBED_MODEL} | Claude: {CLAUDE_MODEL} | cache: {CACHE_MODO}")

## Tarefa 1 – Chaves e clientes

A chave **não fica escrita no código** (lida de `ANTHROPIC_API_KEY`, do *secret* do Colab, ou pedida por teclado) e o ChromaDB recebe **embeddings multilíngues** calculados por nós, em vez do modelo padrão do Chroma, que rende pior em português.

In [ ]:
api_key = os.getenv('ANTHROPIC_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        api_key = getpass.getpass('ANTHROPIC_API_KEY: ')
claude_client = Anthropic(api_key=api_key)

model = SentenceTransformer(EMBED_MODEL)

chroma_client = PersistentClient(path=CHROMA_DIR)
nombre_coleccion = 'documentos_pdf_' + re.sub(r'[^a-zA-Z0-9]+', '_', EMBED_MODEL).strip('_')   # modelo no nome: nunca mistura dimensões
coleccion = chroma_client.get_or_create_collection(
    name=nombre_coleccion,
    metadata={'hnsw:space': 'cosine'}
)
print(f"Coleção '{nombre_coleccion}' com {coleccion.count()} blocos.")

## Tarefa 2 – Extração, limpeza e calibração de tokens

`PdfReader` página a página, para **todos os arquivos da pasta**. A limpeza tira o que custa tokens sem agregar: cabeçalhos/rodapés repetidos (`Manual … • Página 3`), números de página, hifenização, linhas decorativas, base64, linhas de log repetidas e, nesta versão, as **tags HTML de formatação** (`<b>1.</b>` vira `1.`), que o seu PDF carregava em todas as linhas.

No final, `calibrar_tokens` pede à API (endpoint gratuito `count_tokens`) a contagem real de uma amostra do texto e ajusta `CHARS_POR_TOKEN`. No teste com as 100 receitas a estimativa antiga errava por ~1,7×, o que fazia o orçamento de contexto virar quase o dobro do esperado.

In [ ]:
os.makedirs(DOCS_DIR, exist_ok=True)

if CREAR_EJEMPLOS and not glob.glob(os.path.join(DOCS_DIR, '*')):
    ejemplos = {
        'manual_produto.md': """# Manual do produto — Cafeteira Aroma X

Além disso, a Cafeteira Aroma X possui garantia legal de 12 meses a partir da data de compra, mediante apresentação da nota fiscal. Vale destacar que a garantia cobre defeitos de fabricação e não cobre danos por mau uso, quedas ou uso de peças não originais.

É importante notar que, para acionar a garantia, o cliente deve contatar uma assistência técnica autorizada ou o SAC pelo telefone 0800-000-0000. O prazo para conserto é de até 30 dias corridos.""",
    }
    for nombre, texto in ejemplos.items():
        with open(os.path.join(DOCS_DIR, nombre), 'w', encoding='utf-8') as f:
            f.write(texto)
    print('Pasta vazia: criei um documento de exemplo.')


MODIF_DEPOIS = {'\u02dc': '\u0303', '\u02c6': '\u0302', '\u00b4': '\u0301', '`': '\u0300'}   # aplicam-se à letra seguinte


def corrigir_acentos(texto):
    # a Cartilha grava cedilha e til como marcas separadas (c¸ ˜ao): junta cada marca à letra certa
    texto = texto.replace('\u0131', 'i')
    texto = re.sub(r'(\S)\u00b8', lambda m: unicodedata.normalize('NFC', m.group(1) + '\u0327'), texto)
    texto = re.sub(r'([\u02dc\u02c6\u00b4`])([A-Za-z])',
                   lambda m: unicodedata.normalize('NFC', m.group(2) + MODIF_DEPOIS[m.group(1)]), texto)
    return unicodedata.normalize('NFKC', texto)   # NFKC também desfaz ligaduras (ﬁ -> fi)


def quitar_encabezados_pdf(paginas):
    # remove linhas de borda (2 primeiras/2 últimas) que se repetem em >= metade das páginas
    if len(paginas) < 3:
        return paginas
    mascara = lambda l: re.sub(r'\d+', '#', l.strip())

    def indices_borde(p):
        ls = p.split('\n')
        idx = [i for i, l in enumerate(ls) if l.strip()]
        return set(idx[:2] + idx[-2:]) if len(idx) >= 8 else set()

    cuenta = Counter()
    for p in paginas:
        ls = p.split('\n')
        cuenta.update({mascara(ls[i]) for i in indices_borde(p)})
    repetidas = {l for l, c in cuenta.items() if c >= max(2, len(paginas) // 2)}

    salida = []
    for p in paginas:
        ls, bordes = p.split('\n'), indices_borde(p)
        salida.append('\n'.join(l for i, l in enumerate(ls) if not (i in bordes and mascara(l) in repetidas)))
    return salida


def limpiar(texto):
    texto = texto.replace('\r\n', '\n')
    texto = re.sub(r'</?(?:b|i|u|strong|em|br|p|span|div|font|sup|sub)\b[^>]*>', '', texto, flags=re.I)   # tags HTML de formatação
    texto = re.sub(r'(?im)^[^\n]{0,80}[•|–—-]\s*p[áa]g(?:ina)?\.?\s*\d+\s*$', '', texto)              # "Título • Página 3"
    texto = re.sub(r'(\w)-\n(\w)', r'\1\2', texto)                                   # hifenização de PDF
    texto = re.sub(r'[A-Za-z0-9+/=]{200,}', '[blob-removido]', texto)                # base64 e similares
    texto = re.sub(r'(?im)^\s*(p[áa]g(ina)?\.?|page)?\s*\d+(\s*(/|de|of)\s*\d+)?\s*$', '', texto)  # nº de página
    texto = re.sub(r'(?m)^\s*([-_=*]{3,}|\.{4,})\s*$', '', texto)                     # linhas decorativas
    texto = re.sub(r'(?<![.!?:;\n])\n(?=[a-zà-ÿ])', ' ', texto)                       # junta quebra "dura" de PDF

    # colapsa linhas consecutivas idênticas (logs repetitivos)
    lineas, previa, rep = [], None, 0
    for l in texto.split('\n'):
        if l.strip() and l == previa:
            rep += 1
            continue
        if rep:
            lineas[-1] += f'  [repetida {rep + 1}x]'
            rep = 0
        lineas.append(l)
        previa = l
    if rep:
        lineas[-1] += f'  [repetida {rep + 1}x]'
    texto = '\n'.join(lineas)

    texto = re.sub(r'[ \t]+', ' ', texto)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()


def extraer_texto(ruta):
    # PDF: página a página (como no exemplo) já sem cabeçalhos/rodapés repetidos; demais formatos: texto puro
    if ruta.lower().endswith('.pdf'):
        with pymupdf.open(ruta) as pdf:
            paginas = [p.get_text() for p in pdf]
        return '\n\n'.join(quitar_encabezados_pdf(paginas))
    with open(ruta, encoding='utf-8', errors='replace') as f:
        return f.read()


archivos = sorted(
    f for ext in ('pdf', 'txt', 'md', 'log')
    for f in glob.glob(os.path.join(DOCS_DIR, f'*.{ext}'))
)

documentos = [limpiar(corrigir_acentos(extraer_texto(r))) for r in archivos]
titulos_documentos = [os.path.splitext(os.path.basename(r))[0] for r in archivos]


def calibrar_tokens(texto, n=15000):
    # compara caracteres x tokens reais de uma amostra (count_tokens não gera nem cobra saída)
    muestra = texto[:n]
    if len(muestra) < 500:
        return CHARS_POR_TOKEN
    try:
        r = claude_client.messages.count_tokens(model=CLAUDE_MODEL, messages=[{'role': 'user', 'content': muestra}])
        return round(len(muestra) / max(1, r.input_tokens - 10), 2)
    except Exception as e:
        print(f'[Aviso] count_tokens indisponível ({e}); mantendo {CHARS_POR_TOKEN} caracteres/token.')
        return CHARS_POR_TOKEN


CHARS_POR_TOKEN = calibrar_tokens('\n\n'.join(documentos))
print(f"Calibração: {CHARS_POR_TOKEN} caracteres por token")
print(f"Total de documentos carregados: {len(documentos)}")
for t, d in zip(titulos_documentos, documentos):
    print(f"  - {t}: ~{tokens_aprox(d)} tokens")

## Tarefa 3 – Chunking por seção e indexação no ChromaDB

No teste, o `RecursiveCharacterTextSplitter` cortava as receitas no meio (o pudim ficou nos blocos #42, #43 e #44) e os trechos chegavam ao Claude sem título. Agora:

- se o documento tem **seções numeradas em sequência** (`1. Pão de queijo`, `2. Panqueca…`), cada seção vira **um bloco**, com número, título e categoria. A categoria vem do índice do próprio documento (`Sobremesas — 10 receitas`);
- para cada categoria é criado um **bloco de índice** com os títulos, o que permite responder "quais são as sobremesas?" sem depender de similaridade;
- documentos sem esse padrão caem no `RecursiveCharacterTextSplitter` do exemplo (`keep_separator='end'`, sem perder o ponto final);
- o **id é um hash do conteúdo**: se você trocar ou editar um PDF, só os blocos novos são vetorizados e os antigos saem do índice.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
    keep_separator='end',
)

# Cabeçalhos de seção: "Art. 48" (LGPD); "2.3 Phishing" ou "2.3" com o título na linha seguinte (Cartilha)
RE_ART = re.compile(r'^Art\.\s*(\d{1,3})\s*(?:[º°o]\b)?')
RE_SECAO = re.compile(r'^(\d{1,2}(?:\.\d{1,2}){0,2})\s+([^\n.]{3,80}?)\s*$')
RE_NUMERO_SOZINHO = re.compile(r'^\d{1,2}(?:\.\d{1,2}){0,2}$')
RE_CAPITULO = re.compile(r'^CAP[ÍI]TULO\s+([IVXLC]+)\b.*$', re.I)


def titulo_valido(t):
    # título curto, sem ponto (descarta sumário "...... 12" e frases de referência)
    return 3 <= len(t) <= 80 and '.' not in t and not t[0].isdigit()


def sequencia_crescente(cands):
    # maior subsequência com números crescentes: descarta referências fora de ordem
    n = len(cands)
    if n == 0:
        return []
    tam, ant = [1] * n, [-1] * n
    for j in range(n):
        for i in range(j):
            if cands[i][1] < cands[j][1] and tam[i] + 1 > tam[j]:
                tam[j], ant[j] = tam[i] + 1, i
    fim = max(range(n), key=lambda k: tam[k])
    saida = []
    while fim != -1:
        saida.append(cands[fim])
        fim = ant[fim]
    return saida[::-1]


def achar_secoes(linhas):
    # devolve [(linha, chave, titulo, categoria)] ou None se houver menos de 3 seções
    cands, categoria, i = [], '', 0
    while i < len(linhas):
        s = linhas[i].strip()
        i += 1
        if RE_CAPITULO.match(s):
            categoria = s
            continue
        m_art = RE_ART.match(s)
        if m_art:
            chave, titulo, inicio = (int(m_art.group(1)),), f"Art. {m_art.group(1)}", i - 1
        elif RE_NUMERO_SOZINHO.match(s):
            prox = next((x.strip() for x in linhas[i:i + 2] if x.strip()), '')
            if not titulo_valido(prox):
                continue
            chave, titulo, inicio = tuple(int(x) for x in s.split('.')), f"{s} {prox}", i - 1
        else:
            m_sec = RE_SECAO.match(s)
            if not m_sec or not titulo_valido(m_sec.group(2).strip()):
                continue
            chave = tuple(int(x) for x in m_sec.group(1).split('.'))
            titulo, inicio = f"{m_sec.group(1)} {m_sec.group(2).strip()}", i - 1
        cands.append((inicio, chave, titulo, categoria))
    marcas = sequencia_crescente(cands)
    return marcas if len(marcas) >= 3 else None


def dividir_documento(texto, nome):
    linhas = texto.split('\n')
    marcas = achar_secoes(linhas)
    registros = []
    if marcas is None:                                   # sem seções: texto corrido
        for orden, bloque in enumerate(text_splitter.split_text(texto)):
            registros.append({'titulo': nome, 'seccion': '', 'categoria': '', 'referencia': '', 'tipo': 'texto',
                              'orden': orden, 'texto': bloque})
        return registros

    orden = 0
    prefacio = '\n'.join(linhas[:marcas[0][0]]).strip()
    for bloque in text_splitter.split_text(prefacio) if prefacio else []:
        registros.append({'titulo': nome, 'seccion': 'Apresentação', 'categoria': '', 'referencia': '', 'tipo': 'texto',
                          'orden': orden, 'texto': bloque})
        orden += 1
    limites = [m[0] for m in marcas] + [len(linhas)]
    for k, (i, chave, titulo, categoria) in enumerate(marcas):
        corpo = '\n'.join(linhas[i:limites[k + 1]]).strip()
        ref = '.'.join(str(x) for x in chave)
        partes = [corpo] if len(corpo) <= CHUNK_SIZE * 2.5 else text_splitter.split_text(corpo)
        for j, parte in enumerate(partes):
            registros.append({'titulo': nome, 'seccion': titulo, 'categoria': categoria, 'referencia': ref,
                              'tipo': 'seccion', 'orden': orden,
                              'texto': parte if j == 0 else f"{titulo} (cont.)\n{parte}"})
            orden += 1
    return registros


registros = []
for doc, titulo in zip(documentos, titulos_documentos):
    bloques = dividir_documento(doc, titulo)
    registros += bloques
    print(f"  - {titulo}: {len(bloques)} blocos ({sum(b['tipo'] == 'seccion' for b in bloques)} blocos de seção ou artigo)")

ids_chunks = [hashlib.sha1(f"{r['titulo']}|{r['seccion']}|{r['texto']}".encode()).hexdigest()[:20] for r in registros]

existentes = set(coleccion.get()['ids'])
obsoletos = list(existentes - set(ids_chunks))
if obsoletos:
    coleccion.delete(ids=obsoletos)

nuevos = [i for i, id_ in enumerate(ids_chunks) if id_ not in existentes]
if nuevos:
    coleccion.upsert(
        ids=[ids_chunks[i] for i in nuevos],
        embeddings=model.encode([registros[i]['texto'] for i in nuevos], normalize_embeddings=True).tolist(),
        documents=[registros[i]['texto'] for i in nuevos],
        metadatas=[{k: registros[i][k] for k in ('titulo', 'seccion', 'categoria', 'referencia', 'tipo', 'orden')} for i in nuevos],
    )

print(f"Blocos gerados: {len(registros)} | novos indexados: {len(nuevos)} | removidos: {len(obsoletos)} | total no Chroma: {coleccion.count()}")

## Tarefa 4 – Busca híbrida (e decisão: usar contexto ou não)

No teste, a pergunta sobre moqueca trouxe pão, pastel e feijoada com similaridade 0,59: num manual só de receitas **tudo se parece**, e o embedding sozinho não distingue "receita parecida" de "receita certa". A busca agora combina:

1. **Embeddings** (ChromaDB): candidatos por significado.
2. **Cobertura de palavras-chave**: que fração das palavras da pergunta, **pesadas por raridade**, aparece no bloco. "ingredientes" está em todos os blocos e pesa ~0; "moqueca" é rara e decide. Se a palavra rara não aparece em nenhum bloco, o bloco é descartado.
3. **Busca por número**: "receita número 61" vira consulta direta pelos metadados (embedding não entende números).
4. **Perguntas de lista** ("quais são as sobremesas") ganham um bônus nos blocos de índice.

Se nada passar nos filtros, a função devolve `[]` e a pergunta segue **sem contexto** (a Tarefa 7 avisa que a resposta não veio dos documentos). A segunda célula mostra, para as suas perguntas, a similaridade, a cobertura e a rota escolhida, para calibrar `SIM_MINIMA` e `LEX_MIN`.

In [ ]:
sin_acentos = lambda s: ''.join(ch for ch in unicodedata.normalize('NFD', s.lower()) if unicodedata.category(ch) != 'Mn')
STOP = set(sin_acentos(w) for w in """a o as os um uma uns umas de do da dos das em no na nos nas por para com sem sobre que e é ou se ao aos
qual quais como são ser foi tem ter mais meu minha seu sua isso esse essa este esta você voce me te vc pode poderia fazer faz
preciso queria quero gostaria saber diga olá oi the of and to in is are for on with""".split())
RE_REFERENCIA = re.compile(r'\b(?:artigo|art\.?|receita|seção|secao|item|n[º°o]\.?|número|numero|#)\s*(?:n[úu]mero|n[º°o]\.?|#)?\s*(\d{1,3}(?:\.\d{1,2}){0,2})\b', re.I)
QUIERE_LISTA = re.compile(r'\b(quais|lista|liste|listar|todas|todos|quantas|quantos)\b', re.I)
IDF_TECHO = 3.5


def terminos(texto):
    texto = sin_acentos(texto).replace('wi-fi', 'wifi').replace('wi fi', 'wifi')   # 'Wi-Fi' não some por ter 2 letras
    palavras = re.findall(r'[a-z0-9]+', texto)
    return {w.rstrip('s')[:6] for w in palavras if len(w) > 2 and w not in STOP}


def preparar_lexico():
    global DOCS_LEX, IDF, IDF_DESCONOCIDO
    datos = coleccion.get(include=['documents', 'metadatas'])
    DOCS_LEX = {id_: terminos(f"{m.get('seccion', '')} {m.get('categoria', '')} {d}")
                for id_, d, m in zip(datos['ids'], datos['documents'], datos['metadatas'])}
    N = max(1, len(DOCS_LEX))
    df = Counter(t for s in DOCS_LEX.values() for t in s)
    IDF = {t: math.log((N + 1) / (c + 0.5)) for t, c in df.items()}
    IDF_DESCONOCIDO = math.log((N + 1) / 0.5)


def cobertura(q_terms, id_):
    if not q_terms:
        return 1.0
    pesos = {t: min(IDF.get(t, IDF_DESCONOCIDO), IDF_TECHO) for t in q_terms}
    total = sum(pesos.values())
    if total < 0.5:                       # só palavras genéricas, presentes em quase todos os blocos
        return 1.0
    return sum(w for t, w in pesos.items() if t in DOCS_LEX.get(id_, ())) / total


def _rec(id_, doc, meta, sim, cov=0.0, score=0.0):
    return {'id': id_, 'texto': doc, 'titulo': meta['titulo'], 'seccion': meta.get('seccion', ''),
            'categoria': meta.get('categoria', ''), 'orden': meta.get('orden', 0), 'tipo': meta.get('tipo', 'texto'),
            'similitud': sim, 'cobertura': cov, 'score': score}


def puntuar(pregunta):
    n = min(POOL_CANDIDATOS, coleccion.count())
    res = coleccion.query(query_embeddings=model.encode([pregunta], normalize_embeddings=True).tolist(), n_results=n,
                          include=['documents', 'metadatas', 'distances'])
    q_terms, lista = terminos(pregunta), bool(QUIERE_LISTA.search(pregunta))
    cands = []
    for id_, d, m, dist in zip(res['ids'][0], res['documents'][0], res['metadatas'][0], res['distances'][0]):
        sim, cov = 1 - dist, cobertura(q_terms, id_)
        bonus = 0.25 if (lista and m.get('tipo') == 'indice') else 0.0
        cands.append(_rec(id_, d, m, sim, cov, sim + PESO_LEXICO * cov + bonus))
    return sorted(cands, key=lambda c: c['score'], reverse=True)


def recuperar(pregunta, top_k=TOP_K, hibrido=True):
    if coleccion.count() == 0:
        return []
    if not hibrido:                       # como no exemplo simples: sempre devolve os top-k por embedding
        res = coleccion.query(query_embeddings=model.encode([pregunta], normalize_embeddings=True).tolist(),
                              n_results=min(top_k, coleccion.count()), include=['documents', 'metadatas', 'distances'])
        return [_rec(i, d, m, 1 - dist) for i, d, m, dist in zip(res['ids'][0], res['documents'][0], res['metadatas'][0], res['distances'][0])]

    m = RE_REFERENCIA.search(pregunta)   # "artigo 48", "seção 2.3": consulta direta pelos metadados
    if m:
        datos = coleccion.get(where={'$and': [{'referencia': m.group(1)}, {'tipo': 'seccion'}]}, include=['documents', 'metadatas'])
        if datos['ids']:
            return [_rec(i, d, md_, 1.0, 1.0, 2.0) for i, d, md_ in zip(datos['ids'], datos['documents'], datos['metadatas'])][:top_k]

    cands = puntuar(pregunta)
    lista = bool(QUIERE_LISTA.search(pregunta))
    ok = [c for c in cands if (c['similitud'] >= SIM_MINIMA and c['cobertura'] >= LEX_MIN) or c['similitud'] >= SIM_ALTA
          or (lista and c['tipo'] == 'indice' and c['cobertura'] >= LEX_MIN and c['similitud'] >= SIM_MINIMA / 2)]  # listagens: o bloco de índice basta com sim. menor
    if not ok:
        return []
    vistos, unicos = set(), []
    for c in ok:                       # no máximo um bloco por seção, na ordem de relevância
        chave = (c['titulo'], c['seccion']) if c['seccion'] else c['id']
        if chave not in vistos:
            vistos.add(chave)
            unicos.append(c)
    return [c for c in unicos if c['score'] >= unicos[0]['score'] - MARGEN_RELATIVO][:top_k]


preparar_lexico()

pergunta_usuario = "O que diz o art. 48 da LGPD sobre incidentes?"
recuperados = recuperar(pergunta_usuario)

print(f"Pergunta: {pergunta_usuario}\n")
for r in recuperados:
    print(f"{r['categoria']} › {r['seccion'] or r['titulo']}: similitud={r['similitud']:.3f} cobertura={r['cobertura']:.2f} (~{tokens_aprox(r['texto'])} tokens)")
if not recuperados:
    print('Nada relevante: a pergunta seguiria sem contexto.')

In [ ]:
# Calibração: o que a busca decide para cada pergunta (edite as listas com perguntas suas)
perguntas_calibracao = {
    'dentro': ['O que diz o art. 48 da LGPD?', 'Quais cuidados com senhas a Cartilha recomenda?',
               'Como me proteger de phishing?', 'O que é a seção 2.3?', 'Quais são os direitos do titular de dados?'],
    'conferir (está nos documentos?)': ['Quais são as obrigações do controlador em caso de incidente?'],
    'fora':   ['Qual a política de férias da empresa?', 'Explique a diferença entre TCP e UDP', 'Qual a receita de bolo de cenoura?'],
}
filas = []
for grupo, qs in perguntas_calibracao.items():
    for q in qs:
        if coleccion.count():
            top = puntuar(q)[0]
            rec = recuperar(q)
            filas.append({'grupo': grupo, 'pergunta': q, 'sim_max': round(top['similitud'], 3), 'cobertura': round(top['cobertura'], 2),
                          'rota': 'contexto' if rec else 'direta', 'primeiro': (rec[0]['seccion'] if rec else '-')})
display(pd.DataFrame(filas))

## Tarefa 5 – Contexto enxuto e pergunta limpa

**Contexto.** No teste, a compressão por frase soltava ingredientes e passos sem o nome da receita, e a remoção global de duplicatas podia apagar um ingrediente legítimo que se repete entre receitas ("1 colher de sal"). Agora:

- documentos **estruturados** (seções numeradas): cada seção entra **inteira**, com o rótulo `[Documento › Categoria › Seção]`, na ordem de relevância; se não couber no `PRESUPUESTO_CONTEXTO`, as seções menos relevantes são omitidas inteiras. Linhas curtas só são deduplicadas **dentro da mesma seção**;
- **texto corrido**: continua a seleção por frases (muletas como "Além disso," removidas, duplicatas eliminadas, frases mais parecidas com a pergunta quando não cabe tudo).

**Pergunta.** Remove saudações e cortesias, sem inserir lixo na pontuação e mantendo o sentido interrogativo.

In [ ]:
# ============================================================
# PERGUNTA: limpeza antes do RAG
# ============================================================

RELLENO_INICIO = [
    r'^\s*(olá|ola|oi|bom dia|boa tarde|boa noite)\b[,!.\s]*',                                    # saudações
    r'^\s*((você|voce|vc)\s+)?(poderia|poderias|pode|podes)\s+(me\s+)?'
    r'(dizer|explicar|informar|ajudar(?:\s+a)?|mostrar|indicar)\s+',                              # "poderia me dizer..."
    r'^\s*((você|voce|vc)\s+)?(consegue|conseguiria|conseguem)\s+(me\s+)?'
    r'(dizer|explicar|informar|mostrar|ajudar(?:\s+a)?|encontrar)\s+',                            # "você consegue me mostrar..."
    r'^\s*(eu\s+)?(gostaria|queria|quero|desejo)\s+(de\s+)?(saber|entender|descobrir|verificar)\s+(se\s+)?',   # "gostaria de saber..."
    r'^\s*me\s+(diga|diz|fale|fala|explique|explica|informe|informa|mostre|mostra)\s+',           # "me diga..."
    r'^\s*(eu\s+)?(gostaria|queria|quero)\s+(que\s+)?(você|voce|vc)\s+(me\s+)?'
    r'(dissesse|explicasse|informasse|mostrasse)\s+',                                             # "gostaria que você explicasse..."
    r'^\s*(você|voce|vc)?\s*(pode|poderia|consegue)\s+me\s+ajudar\s+(a\s+)?',                     # "pode me ajudar a..."
]

RELLENO_EM_MEIO = (r'\b(por favor|por gentileza|por obséquio|se possível|desde já agradeço|'
                   r'muito obrigad[oa]|obrigad[oa])\b[,.!]?\s*')

RELLENO_FIM = [
    r'\s*[,.!]?\s*(por favor|por gentileza)\s*[.!?]*$',
    r'\s*[,.!]?\s*(obrigado|obrigada|valeu|agradeço)\s*[.!?]*$',
    r'\s*[.!]?\s*(desde já agradeço|muito obrigado|muito obrigada)\s*[.!?]*$',
]

FRASES_VAZIAS = {"oi", "ola", "olá", "bom dia", "boa tarde", "boa noite", "obrigado", "obrigada", "valeu", "tudo bem"}
INTERROGATIVOS = ('como', 'qual', 'quais', 'quanto', 'quantos', 'quantas', 'quando', 'onde', 'por que', 'o que', 'quem')


def limpar_pergunta(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r'<[^>]+>', ' ', texto.strip())
    for _ in range(3):                                    # repete: "Oi! Por favor, poderia me dizer..." precisa de mais de uma passada
        antes = texto
        for padrao in RELLENO_INICIO:
            texto = re.sub(padrao, '', texto, flags=re.IGNORECASE)
        texto = re.sub(RELLENO_EM_MEIO, ' ', texto, flags=re.IGNORECASE).strip(' ,;')
        if texto == antes:
            break
    texto = re.sub(RELLENO_EM_MEIO, ' ', texto, flags=re.IGNORECASE)
    for padrao in RELLENO_FIM:
        texto = re.sub(padrao, '', texto, flags=re.IGNORECASE)
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'\s+([,.!?;:])', r'\1', texto)
    texto = re.sub(r'([!?.,])\1+', r'\1', texto)          # pontuação repetida -> uma só
    texto = re.sub(r'\s*,\s*([?!.])', r'\1', texto)       # "cenoura,?" -> "cenoura?"
    return texto.strip(' ,;')


def pergunta_valida(texto):
    return bool(texto) and texto.lower().strip() not in FRASES_VAZIAS and len(texto.split()) >= 2


def comprimir_pregunta(pergunta):
    if not isinstance(pergunta, str):
        return ""
    original = pergunta.strip()
    q = limpar_pergunta(original)
    if not pergunta_valida(q):                            # limpeza destruiu a pergunta: fica com a original
        return original
    pedia_saber = re.search(r'\b(saber|entender|descobrir|verificar)\b', original, re.I)
    if (pedia_saber or q.lower().startswith(INTERROGATIVOS)) and not q.endswith('?'):
        q = q.rstrip('.!,') + '?'                         # mantém o sentido interrogativo
    return q[0].upper() + q[1:]


perguntas_teste = [
    "Olá, poderia me dizer o que a LGPD exige em caso de incidente de segurança?",
    "Você pode me explicar como criar uma senha forte, por favor?",
    "Eu gostaria de saber o que é phishing.",
    "Me informe o que diz o artigo 46, obrigada!",
    "Por favor, qual é a seção 8.2?",
    "Oi! Quero saber quais são as regras para alteração de senhas.",
    "Explique a diferença entre TCP e UDP",
]
for pergunta in perguntas_teste:
    print(f"Original: {pergunta}\nLimpa   : {comprimir_pregunta(pergunta)}\n{'-' * 70}")

In [ ]:
# ============================================================
# CONTEXTO
# ============================================================

MULETILLAS = [
    r'al[ée]m disso', r'ademais', r'por outro lado', r'em outras palavras', r'em resumo', r'basicamente', r'de modo geral',
    r'vale (a pena )?(destacar|ressaltar|mencionar|lembrar) que', r'é importante (notar|destacar|ressaltar|lembrar) que',
    r'cabe (destacar|ressaltar) que', r'(observe|note) que', r'como (mencionado|dito) (anteriormente|acima)',
    r'además', r'por otro lado', r'en otras palabras', r'en resumen', r'básicamente', r'en general',
    r'cabe (destacar|señalar) que', r'es importante (notar|destacar|señalar|recordar) que',
    r'moreover', r'furthermore', r'in addition', r'in other words', r'basically', r'it is important to note that',
]
RE_MULETILLAS = re.compile(r'^(?:' + '|'.join(MULETILLAS) + r')\b[,:]?\s*', re.I)
RE_ARTICULOS = re.compile(r'\b(o|os|as|um|uma|uns|umas|el|los|las|un|una|unos|unas|the|an)\b\s*', re.I)   # 'a'/'la' ficam: ambíguos


def limpiar_linea(linea):
    linea = re.sub(r'^\s*[•*]\s*', '- ', linea.strip())
    if not re.match(r'^(-|\d+[.)])\s', linea):                   # muletas só em linhas de prosa
        linea = RE_MULETILLAS.sub('', linea).strip()
        linea = linea[:1].upper() + linea[1:]
    if ELIMINAR_ARTICULOS:
        linea = RE_ARTICULOS.sub('', linea)
    return re.sub(r'\s+', ' ', linea).strip()


def etiqueta(r):
    return '[' + ' › '.join(x for x in (r['titulo'], r.get('categoria', ''), r.get('seccion', '')) if x) + ']'


def comprimir_secciones(chunks_rec, presupuesto):
    # Seções inteiras, com rótulo; duplicatas curtas só dentro da mesma seção, longas em todo o contexto.
    largas_vistas, bloques = set(), []
    for r in chunks_rec:
        vistas, lineas = set(), []
        for l in r['texto'].split('\n'):
            l = limpiar_linea(l)
            clave = re.sub(r'\W+', ' ', l).strip().lower()
            if not l or (len(clave) < 60 and clave in vistas) or (len(clave) >= 60 and clave in largas_vistas):
                continue
            (largas_vistas if len(clave) >= 60 else vistas).add(clave)
            lineas.append(l)
        texto = etiqueta(r) + '\n' + '\n'.join(lineas)
        bloques.append((texto, tokens_aprox(texto)))
    elegidos, usado = [], 0
    for texto, costo in bloques:                                # já vêm por relevância
        if not elegidos or usado + costo <= presupuesto:
            elegidos.append(texto)
            usado += costo
    return '\n\n'.join(elegidos)


def dividir_frases(texto):
    frases = []
    for linea in texto.split('\n'):
        linea = re.sub(r'^\s*(#{1,6}|[-*•]|\d+[.)])\s*', '', linea).strip()
        if not linea:
            continue
        partes = re.split(r'(?<=[.!?;])\s+(?=[A-ZÁÀÂÃÉÊÍÓÔÕÚÇÑ¿¡"“(\d])', linea)
        frases += [p.strip() for p in partes if len(p.strip()) > 2]
    return frases


def comprimir_frases(chunks_rec, pregunta, presupuesto):
    # Texto corrido: frases limpas, sem duplicatas; se não couber, as mais parecidas com a pergunta.
    rango_doc, items, vistas = {}, [], set()
    for r in chunks_rec:
        rango_doc.setdefault(r['titulo'], len(rango_doc))
        for i, f in enumerate(dividir_frases(r['texto'])):
            f = RE_MULETILLAS.sub('', f).strip()
            f = f[:1].upper() + f[1:]
            clave = re.sub(r'\W+', ' ', f).strip().lower()
            if not f or clave in vistas:
                continue
            vistas.add(clave)
            items.append({'titulo': r['titulo'], 'orden': r['orden'], 'i': i, 'frase': f})
    costo = lambda it: tokens_aprox(it['frase']) + 2
    if sum(costo(it) for it in items) > presupuesto:
        vq = model.encode([pregunta], normalize_embeddings=True)[0]
        vf = model.encode([it['frase'] for it in items], normalize_embeddings=True)
        for it, s in zip(items, vf @ vq):
            it['sim'] = float(s)
        usado, elegidos = 0, []
        for it in sorted(items, key=lambda x: x['sim'], reverse=True):
            if usado + costo(it) <= presupuesto:
                elegidos.append(it)
                usado += costo(it)
        items = elegidos
    items.sort(key=lambda it: (rango_doc[it['titulo']], it['orden'], it['i']))
    salida, actual = [], None
    for it in items:
        if it['titulo'] != actual:
            salida.append(f"[{it['titulo']}]")
            actual = it['titulo']
        salida.append(f"- {it['frase']}")
    return '\n'.join(salida)


def comprimir_en_topicos(chunks_rec, pregunta, presupuesto=PRESUPUESTO_CONTEXTO):
    if not chunks_rec:
        return ''
    if all(r.get('seccion') for r in chunks_rec):
        return comprimir_secciones(chunks_rec, presupuesto)
    return comprimir_frases(chunks_rec, pregunta, presupuesto)


# --- Demonstração ---
bruto_ctx = '\n\n'.join(r['texto'] for r in recuperados)
topicos = comprimir_en_topicos(recuperados, pergunta_usuario)
print(topicos or '(sem contexto para esta pergunta)')
print(f"\nTokens estimados: blocos brutos={tokens_aprox(bruto_ctx)} -> contexto enviado={tokens_aprox(topicos)}")

## Tarefa 6 – Chamada ao Claude com prompt caching

Mesma estrutura do exemplo: **contexto primeiro** (prefixo que pode ficar em cache) e **pergunta depois**. Pontos desta versão:

- `cache_control` só é aplicado quando faz sentido (`CACHE_MODO='auto'`): contexto com `CACHE_MIN_TOKENS` ou mais **e** idêntico ao da chamada anterior. `'siempre'` marca sempre e `'nunca'` desliga.
- **Métrica correta:** com cache, `usage.input_tokens` conta só os tokens fora do cache. O total do prompt é `input + cache_creation + cache_read`, e o custo equivalente é `input + 1.25×creation + 0.1×read`.
- **Saída:** no teste, a saída cobrada era ~2× o texto visível. A função devolve os tipos de bloco da resposta e avisa se houver `thinking`; nesse caso suba `CLAUDE_MAX_TOKENS` ou use `DESACTIVAR_THINKING=True`. Também avisa se a resposta foi cortada por `max_tokens`.
- O *system prompt* manda o Claude dizer **"isso não consta nos trechos recuperados"** quando a pergunta é sobre os documentos e o contexto não responde, em vez de completar em silêncio com conhecimento geral.

In [ ]:
SISTEMA_CLAUDE = (
    'Responda em português, de forma direta (até ~200 palavras). '
    'Pode vir um CONTEXTO com trechos de documentos de segurança e privacidade, cada um precedido de [Documento › Categoria › Seção]. '
    'Se o contexto responder à pergunta, baseie-se nele e cite a seção (por exemplo, "art. 48 da LGPD" ou "seção 8.2 da Cartilha"). '
    'Se a pergunta for claramente sobre os documentos e o contexto não trouxer a resposta, diga primeiro que isso não consta '
    'nos trechos recuperados e marque qualquer complemento como conhecimento geral. '
    'Não invente artigos, prazos, obrigações ou procedimentos. '
    'Se a pergunta não depender dos documentos, responda normalmente, sem mencionar o contexto.'
)

_estado_cache = {'hash': None}


def decidir_cache(contexto, forzar=None):
    if not contexto or CACHE_MODO == 'nunca':
        return False
    if forzar is not None:
        return forzar
    if CACHE_MODO == 'siempre':
        return True
    h = hashlib.sha1(contexto.encode()).hexdigest()          # 'auto': grande E repetido
    repetido = (h == _estado_cache['hash'])
    _estado_cache['hash'] = h
    return repetido and tokens_aprox(contexto) >= CACHE_MIN_TOKENS


def generar_con_claude(pregunta, contexto='', max_tokens=CLAUDE_MAX_TOKENS, forzar_cache=None):
    usar_cache = decidir_cache(contexto, forzar_cache)
    if contexto:
        bloque_ctx = {'type': 'text', 'text': f'Baseie-se no contexto abaixo quando for relevante:\n\n{contexto}'}
        if usar_cache:
            bloque_ctx['cache_control'] = {'type': 'ephemeral'}      # guarda o contexto no cache da Anthropic
        contenido = [bloque_ctx, {'type': 'text', 'text': f'Pergunta do utilizador: {pregunta}'}]
    else:
        contenido = pregunta

    kwargs = dict(model=CLAUDE_MODEL, max_tokens=max_tokens, system=SISTEMA_CLAUDE,
                  messages=[{'role': 'user', 'content': contenido}])
    if TEMPERATURA is not None:
        kwargs['temperature'] = TEMPERATURA
    if DESACTIVAR_THINKING:
        kwargs['thinking'] = {'type': 'disabled'}
    try:
        resp = claude_client.messages.create(**kwargs)
    except Exception as e:
        return {'texto': f'[Erro ao chamar o Claude: {e}]', 'tokens_in': 0, 'tokens_out': 0, 'cache_creation': 0,
                'cache_read': 0, 'costo_in_eq': 0.0, 'cache_usado': usar_cache, 'cortada': False, 'bloques': []}

    u = resp.usage
    normal = u.input_tokens
    creacion = getattr(u, 'cache_creation_input_tokens', 0) or 0
    lectura = getattr(u, 'cache_read_input_tokens', 0) or 0
    return {
        'texto': ''.join(b.text for b in resp.content if b.type == 'text'),
        'tokens_in': normal + creacion + lectura,                   # total do prompt
        'tokens_out': u.output_tokens,
        'cache_creation': creacion,
        'cache_read': lectura,
        'costo_in_eq': normal + 1.25 * creacion + 0.1 * lectura,    # em "tokens normais"
        'cache_usado': usar_cache,
        'cortada': resp.stop_reason == 'max_tokens',
        'bloques': [b.type for b in resp.content],
    }

## Tarefa 7 – Função completa e medição da economia

`preguntar` junta tudo, com `modo`:

- `"filtrado"` (padrão): pergunta limpa → busca híbrida → contexto enxuto (ou pergunta sozinha) → Claude.
- `"completo"`: o fluxo do exemplo simples, para comparar: pergunta original + os top-k por embedding **sem** filtros nem compressão.

Quando nenhum trecho dos documentos passa nos filtros, a resposta sai com a **nota** de que veio do conhecimento geral do Claude, não do manual. `contexto_fijo` permite o modo documento fixo (cache).

In [ ]:
def preguntar(pregunta, modo='filtrado', top_k=TOP_K, contexto_fijo=None, mostrar_contexto=False, verbose=True):
    if contexto_fijo is not None:                      # modo documento fixo
        q, rec, contexto = (comprimir_pregunta(pregunta) if modo == 'filtrado' else pregunta), [], contexto_fijo
        forzar = True
    elif modo == 'filtrado':
        q = comprimir_pregunta(pregunta)
        rec = recuperar(q, top_k)
        contexto, forzar = (comprimir_en_topicos(rec, q) if rec else ''), None
    elif modo == 'completo':
        q = pregunta
        rec = recuperar(q, top_k, hibrido=False)       # como no exemplo: sempre manda os top-k
        contexto, forzar = '\n\n'.join(r['texto'] for r in rec), None
    else:
        raise ValueError('modo deve ser "filtrado" ou "completo"')

    ruta = 'fixo' if contexto_fijo is not None else ('contexto' if contexto else 'direta')
    nota = ''
    if ruta == 'direta' and coleccion.count() > 0:
        nota = 'Nenhum trecho dos documentos passou nos filtros: resposta do conhecimento geral do Claude, não do manual.'
    if verbose:
        print(f"Rota: {ruta} | pergunta enviada: {q!r}")
        for r in rec:
            print(f"  - {r['categoria'] + ' › ' if r['categoria'] else ''}{r['seccion'] or r['titulo']} "
                  f"(sim={r['similitud']:.2f}, cobertura={r['cobertura']:.2f})")
        if mostrar_contexto and contexto:
            print('\n--- CONTEXTO ENVIADO ---')
            print(contexto)

    res = generar_con_claude(q, contexto, forzar_cache=forzar)
    if verbose:
        print(f"\n--- RESPOSTA DO CLAUDE ({modo}) ---")
        if nota:
            print(f"[Nota] {nota}\n")
        print(res['texto'])
        print(f"\n--- Métricas --- prompt total={res['tokens_in']} | saída={res['tokens_out']} | "
              f"cache escrito={res['cache_creation']} | cache lido={res['cache_read']} | "
              f"custo de entrada equivalente={res['costo_in_eq']:.0f} | cache marcado={res['cache_usado']}")
        if 'thinking' in res['bloques']:
            print('[Aviso] A resposta tem bloco de raciocínio (thinking), que conta na saída: ajuste CLAUDE_MAX_TOKENS ou DESACTIVAR_THINKING.')
        if res['cortada']:
            print('[Aviso] Resposta cortada por max_tokens: suba CLAUDE_MAX_TOKENS ou peça uma resposta mais curta.')
    return {**res, 'modo': modo, 'ruta': ruta, 'pregunta_enviada': q, 'nota': nota}


_ = preguntar('Olá, por favor, o que diz o art. 48 da LGPD?', mostrar_contexto=True)

### Loop interativo

Digite `sair` para terminar. Pode perguntar sobre os documentos ou sobre qualquer outra coisa.

In [ ]:
modo_activo = 'filtrado'   # ou "completo"

while True:
    pregunta_usuario = input(f"\n[{modo_activo}] Sua pergunta (ou 'sair' para terminar): ")
    if pregunta_usuario.strip().lower() in ('sair', 'salir', 'exit'):
        break
    preguntar(pregunta_usuario, modo=modo_activo)

### Medição: filtrado × completo

Mesmas perguntas nos dois modos; compara o **custo de entrada equivalente** (já considerando cache) e mostra a rota usada. Atenção ao ler o resultado: uma economia alta numa pergunta com rota `direta` significa que o `filtrado` não achou nada nos documentos (confira se isso é o esperado).

In [ ]:
preguntas_prueba = [
    "Olá, poderia me dizer quais são as bases legais para tratamento de dados?",
    "Você pode me explicar o que é phishing, por favor?",
    "Eu gostaria de saber quais são os direitos do titular de dados.",
    "Por favor, qual é o artigo 46?",
    "Oi! Quero saber quais cuidados tomar com a segurança de redes Wi-Fi.",
    "Qual a política de férias da empresa?",
]

filas = []
for p in preguntas_prueba:
    c = preguntar(p, modo='completo', verbose=False)
    f = preguntar(p, modo='filtrado', verbose=False)
    ahorro = 100 * (1 - f['costo_in_eq'] / c['costo_in_eq']) if c['costo_in_eq'] else 0.0
    filas.append({
        'pergunta': p[:55],
        'rota_filtrado': f['ruta'],
        'entrada_completo': round(c['costo_in_eq']),
        'entrada_filtrado': round(f['costo_in_eq']),
        'economia_%': round(ahorro, 1),
        'saida_completo': c['tokens_out'],
        'saida_filtrado': f['tokens_out'],
        'cortada': f['cortada'],
    })

df = pd.DataFrame(filas)
display(df)
print(f"\nEconomia média no custo de entrada: {df['economia_%'].mean():.1f}%")

### Modo documento fixo: quando o cache compensa

No teste das 100 receitas, mandar o manual inteiro custou ~29,9 mil tokens equivalentes na 1ª pergunta e ~2,4 mil nas seguintes, contra ~1,1 mil por pergunta no RAG: **para este manual o RAG sempre é mais barato**. O documento fixo só vale quando as perguntas precisam do documento todo (resumos, comparações entre várias seções) ou quando o documento é curto e você vai fazer muitas perguntas seguidas. Precisa de ~1.000+ tokens; abaixo disso a API ignora o cache sem avisar.

In [ ]:
# Documento fixo: um trecho curto da LGPD (arts. 46 a 50, incidentes e segurança), não o corpus inteiro
ARTS_FIXOS = {'Art. 46', 'Art. 47', 'Art. 48', 'Art. 49', 'Art. 50'}
texto_fijo = '\n\n'.join(r['texto'] for r in registros
                          if r['titulo'].startswith('Lei_geral') and r['seccion'] in ARTS_FIXOS)
if tokens_aprox(texto_fijo) < CACHE_MIN_TOKENS:
    print(f"Contexto com ~{tokens_aprox(texto_fijo)} tokens: abaixo do mínimo cacheável (~{CACHE_MIN_TOKENS}).")
else:
    print(f"Contexto fixo de ~{tokens_aprox(texto_fijo)} tokens (compare com o custo médio do RAG na tabela acima).")
    for q in ['Quais são as obrigações do controlador em caso de incidente?',
              'Quem deve adotar medidas de segurança para proteger dados pessoais?',
              'Quais boas práticas de governança podem ser criadas?']:
        r = preguntar(q, contexto_fijo=texto_fijo, verbose=False)
        print(f"{q!r}: escrito={r['cache_creation']} lido={r['cache_read']} custo equivalente={r['costo_in_eq']:.0f}")


### Revisão de qualidade

Economizar não vale se a resposta piora. Compare as duas respostas para uma receita (a receita deve sair completa nos dois modos):

In [ ]:
q = 'O que é phishing e como se proteger?'
for modo in ('completo', 'filtrado'):
    print(f"\n========== {modo.upper()} ==========")
    preguntar(q, modo=modo)

## O que mudou nesta versão (documentos de segurança e LGPD)

- **Leitura com PyMuPDF e correção de acentos**, que a Cartilha precisa (cedilha e til vinham como marcas separadas).
- **Um bloco por artigo ou seção**: `Art. 48` da LGPD (65 artigos) e `2.3 Phishing` da Cartilha (69 seções), com referência para a busca direta (`artigo 48`, `seção 2.3`).
- **Ligaduras e cabeçalhos**: a normalização NFKC desfaz ligaduras (`ﬁ` → `fi`), e os cabeçalhos repetidos são removidos página a página.
- **Prompt em português** voltado a políticas e procedimentos: cita a seção e não inventa artigos ou prazos.
- **Os parâmetros de busca (`SIM_MINIMA`, `LEX_MIN`, `PRESUPUESTO_CONTEXTO`) ainda precisam ser calibrados** com as perguntas reais, na célula 14.

In [ ]:
import re
import pandas as pd


def referencia_de(r):
    # referência a partir do rótulo da seção: "Art. 48" -> 48, "2.3 Phishing" -> 2.3
    s = r['seccion'] or ''
    m = re.match(r'^Art\.\s*(\d+)', s) or re.match(r'^(\d+(?:\.\d+)*)\s', s)
    return m.group(1) if m else ''


def acertou(recuperados, doc, esperado):
    for r in recuperados:
        if doc and not r['titulo'].startswith(doc):
            continue
        if referencia_de(r) in esperado:
            return True
    return False


PERGUNTAS_TESTE = [
    {"id": 1, "tipo": 'paráfrase', "pergunta": 'Qual é o objetivo da Lei Geral de Proteção de Dados?', "doc": 'Lei_geral', "esperado": ['1']},
    {"id": 2, "tipo": 'paráfrase', "pergunta": 'Quais são os fundamentos da proteção de dados pessoais?', "doc": 'Lei_geral', "esperado": ['2']},
    {"id": 3, "tipo": 'paráfrase', "pergunta": 'A LGPD se aplica a empresas privadas?', "doc": 'Lei_geral', "esperado": ['3']},
    {"id": 4, "tipo": 'paráfrase', "pergunta": 'A lei vale para quem usa dados só para fins pessoais?', "doc": 'Lei_geral', "esperado": ['4']},
    {"id": 5, "tipo": 'paráfrase', "pergunta": 'Quem é o titular dos dados?', "doc": 'Lei_geral', "esperado": ['5']},
    {"id": 6, "tipo": 'paráfrase', "pergunta": 'O que é um agente de tratamento?', "doc": 'Lei_geral', "esperado": ['5']},
    {"id": 7, "tipo": 'paráfrase', "pergunta": 'O que é anonimização?', "doc": 'Lei_geral', "esperado": ['5']},
    {"id": 8, "tipo": 'paráfrase', "pergunta": 'O que é o princípio da necessidade?', "doc": 'Lei_geral', "esperado": ['6']},
    {"id": 9, "tipo": 'paráfrase', "pergunta": 'O que significa o princípio da transparência?', "doc": 'Lei_geral', "esperado": ['6']},
    {"id": 10, "tipo": 'paráfrase', "pergunta": 'Quais são as bases legais para tratar dados pessoais?', "doc": 'Lei_geral', "esperado": ['7']},
    {"id": 11, "tipo": 'paráfrase', "pergunta": 'O consentimento precisa ser dado por escrito?', "doc": 'Lei_geral', "esperado": ['8']},
    {"id": 12, "tipo": 'paráfrase', "pergunta": 'O titular pode saber como seus dados estão sendo usados?', "doc": 'Lei_geral', "esperado": ['9']},
    {"id": 13, "tipo": 'paráfrase', "pergunta": 'O que é legítimo interesse do controlador?', "doc": 'Lei_geral', "esperado": ['10']},
    {"id": 14, "tipo": 'paráfrase', "pergunta": 'Dados de saúde exigem regras especiais?', "doc": 'Lei_geral', "esperado": ['11']},
    {"id": 15, "tipo": 'paráfrase', "pergunta": 'Dados anonimizados continuam sendo dados pessoais?', "doc": 'Lei_geral', "esperado": ['12']},
    {"id": 16, "tipo": 'paráfrase', "pergunta": 'Como deve ser feito o tratamento de dados de crianças e adolescentes?', "doc": 'Lei_geral', "esperado": ['14']},
    {"id": 17, "tipo": 'paráfrase', "pergunta": 'Quando o tratamento de dados deve terminar?', "doc": 'Lei_geral', "esperado": ['15']},
    {"id": 18, "tipo": 'paráfrase', "pergunta": 'Os dados pessoais precisam ser eliminados depois do uso?', "doc": 'Lei_geral', "esperado": ['16']},
    {"id": 19, "tipo": 'paráfrase', "pergunta": 'Posso pedir ao controlador a confirmação de que trata meus dados?', "doc": 'Lei_geral', "esperado": ['18']},
    {"id": 20, "tipo": 'paráfrase', "pergunta": 'Posso pedir revisão de uma decisão tomada só por sistema automatizado?', "doc": 'Lei_geral', "esperado": ['20']},
    {"id": 21, "tipo": 'paráfrase', "pergunta": 'O titular pode buscar seus direitos na justiça?', "doc": 'Lei_geral', "esperado": ['22']},
    {"id": 22, "tipo": 'paráfrase', "pergunta": 'Quando é permitida a transferência internacional de dados?', "doc": 'Lei_geral', "esperado": ['33']},
    {"id": 23, "tipo": 'paráfrase', "pergunta": 'Preciso manter registro das operações de tratamento?', "doc": 'Lei_geral', "esperado": ['37']},
    {"id": 24, "tipo": 'paráfrase', "pergunta": 'O que é o relatório de impacto à proteção de dados?', "doc": 'Lei_geral', "esperado": ['38']},
    {"id": 25, "tipo": 'paráfrase', "pergunta": 'Qual a responsabilidade do operador no tratamento?', "doc": 'Lei_geral', "esperado": ['39']},
    {"id": 26, "tipo": 'paráfrase', "pergunta": 'O que faz o encarregado pelo tratamento de dados?', "doc": 'Lei_geral', "esperado": ['41']},
    {"id": 27, "tipo": 'paráfrase', "pergunta": 'Quem responde por danos causados pelo tratamento de dados?', "doc": 'Lei_geral', "esperado": ['42']},
    {"id": 28, "tipo": 'paráfrase', "pergunta": 'Em quais casos o agente de tratamento não é responsabilizado?', "doc": 'Lei_geral', "esperado": ['43']},
    {"id": 29, "tipo": 'paráfrase', "pergunta": 'Quando o tratamento de dados é considerado irregular?', "doc": 'Lei_geral', "esperado": ['44']},
    {"id": 30, "tipo": 'paráfrase', "pergunta": 'Quais medidas de segurança devem ser adotadas para proteger dados?', "doc": 'Lei_geral', "esperado": ['46']},
    {"id": 31, "tipo": 'paráfrase', "pergunta": 'Todos que participam do tratamento precisam garantir a segurança da informação?', "doc": 'Lei_geral', "esperado": ['47']},
    {"id": 32, "tipo": 'paráfrase', "pergunta": 'Como os sistemas de tratamento de dados devem ser estruturados?', "doc": 'Lei_geral', "esperado": ['49']},
    {"id": 33, "tipo": 'paráfrase', "pergunta": 'O que são boas práticas de governança para dados pessoais?', "doc": 'Lei_geral', "esperado": ['50']},
    {"id": 34, "tipo": 'paráfrase', "pergunta": 'Quais são as sanções para quem descumpre a lei?', "doc": 'Lei_geral', "esperado": ['52']},
    {"id": 35, "tipo": 'paráfrase', "pergunta": 'O que é a ANPD?', "doc": 'Lei_geral', "esperado": ['55']},
    {"id": 36, "tipo": 'referência', "pergunta": 'O que diz o artigo 18?', "doc": 'Lei_geral', "esperado": ['18']},
    {"id": 37, "tipo": 'referência', "pergunta": 'O que diz o art. 41?', "doc": 'Lei_geral', "esperado": ['41']},
    {"id": 38, "tipo": 'referência', "pergunta": 'Qual o conteúdo do artigo 7?', "doc": 'Lei_geral', "esperado": ['7']},
    {"id": 39, "tipo": 'referência', "pergunta": 'O que diz o artigo 11 sobre dados sensíveis?', "doc": 'Lei_geral', "esperado": ['11']},
    {"id": 40, "tipo": 'paráfrase', "pergunta": 'O que é furto de identidade?', "doc": 'cartilha', "esperado": ['2.1']},
    {"id": 41, "tipo": 'paráfrase', "pergunta": 'O que é fraude de antecipação de recursos?', "doc": 'cartilha', "esperado": ['2.2']},
    {"id": 42, "tipo": 'paráfrase', "pergunta": 'Como funciona o golpe do comércio eletrônico fraudulento?', "doc": 'cartilha', "esperado": ['2.4']},
    {"id": 43, "tipo": 'paráfrase', "pergunta": 'O que é um boato (hoax)?', "doc": 'cartilha', "esperado": ['2.5']},
    {"id": 44, "tipo": 'paráfrase', "pergunta": 'O que é exploração de vulnerabilidades?', "doc": 'cartilha', "esperado": ['3.1']},
    {"id": 45, "tipo": 'paráfrase', "pergunta": 'O que é varredura em redes (scan)?', "doc": 'cartilha', "esperado": ['3.2']},
    {"id": 46, "tipo": 'paráfrase', "pergunta": 'O que é falsificação de e-mail?', "doc": 'cartilha', "esperado": ['3.3']},
    {"id": 47, "tipo": 'paráfrase', "pergunta": 'O que é interceptação de tráfego (sniffing)?', "doc": 'cartilha', "esperado": ['3.4']},
    {"id": 48, "tipo": 'paráfrase', "pergunta": 'O que é ataque de força bruta?', "doc": 'cartilha', "esperado": ['3.5']},
    {"id": 49, "tipo": 'paráfrase', "pergunta": 'O que é desfiguração de página (defacement)?', "doc": 'cartilha', "esperado": ['3.6']},
    {"id": 50, "tipo": 'paráfrase', "pergunta": 'O que é negação de serviço (DoS e DDoS)?', "doc": 'cartilha', "esperado": ['3.7']},
    {"id": 51, "tipo": 'paráfrase', "pergunta": 'O que são worms?', "doc": 'cartilha', "esperado": ['4.2']},
    {"id": 52, "tipo": 'paráfrase', "pergunta": 'O que são bot e botnet?', "doc": 'cartilha', "esperado": ['4.3']},
    {"id": 53, "tipo": 'paráfrase', "pergunta": 'Para que serve um spyware?', "doc": 'cartilha', "esperado": ['4.4']},
    {"id": 54, "tipo": 'paráfrase', "pergunta": 'O que é um backdoor?', "doc": 'cartilha', "esperado": ['4.5']},
    {"id": 55, "tipo": 'paráfrase', "pergunta": 'O que é cavalo de troia (trojan)?', "doc": 'cartilha', "esperado": ['4.6']},
    {"id": 56, "tipo": 'paráfrase', "pergunta": 'O que é rootkit?', "doc": 'cartilha', "esperado": ['4.7']},
    {"id": 57, "tipo": 'paráfrase', "pergunta": 'Como se proteger de códigos maliciosos?', "doc": 'cartilha', "esperado": ['4.8']},
    {"id": 58, "tipo": 'paráfrase', "pergunta": 'Qual a diferença entre vírus, worm e trojan?', "doc": 'cartilha', "esperado": ['4.9']},
    {"id": 59, "tipo": 'paráfrase', "pergunta": 'Quais cuidados devo ter com spam?', "doc": 'cartilha', "esperado": ['5.1']},
    {"id": 60, "tipo": 'paráfrase', "pergunta": 'O que são códigos móveis?', "doc": 'cartilha', "esperado": ['6.2']},
    {"id": 61, "tipo": 'paráfrase', "pergunta": 'Janelas de pop-up podem ser perigosas?', "doc": 'cartilha', "esperado": ['6.3']},
    {"id": 62, "tipo": 'paráfrase', "pergunta": 'Plug-ins e extensões de navegador são seguros?', "doc": 'cartilha', "esperado": ['6.4']},
    {"id": 63, "tipo": 'paráfrase', "pergunta": 'O que são links patrocinados?', "doc": 'cartilha', "esperado": ['6.5']},
    {"id": 64, "tipo": 'paráfrase', "pergunta": 'Quais são os riscos de programas de compartilhamento de arquivos (P2P)?', "doc": 'cartilha', "esperado": ['6.7']},
    {"id": 65, "tipo": 'paráfrase', "pergunta": 'O que é compartilhamento de recursos e quais riscos traz?', "doc": 'cartilha', "esperado": ['6.8']},
    {"id": 66, "tipo": 'paráfrase', "pergunta": 'Quando devo notificar um abuso ou incidente?', "doc": 'cartilha', "esperado": ['7.2']},
    {"id": 67, "tipo": 'paráfrase', "pergunta": 'Com que frequência devo fazer backup?', "doc": 'cartilha', "esperado": ['7.5']},
    {"id": 68, "tipo": 'paráfrase', "pergunta": 'Como os registros de eventos (logs) ajudam na segurança?', "doc": 'cartilha', "esperado": ['7.6']},
    {"id": 69, "tipo": 'paráfrase', "pergunta": 'Preciso de ferramenta antimalware no computador?', "doc": 'cartilha', "esperado": ['7.7']},
    {"id": 70, "tipo": 'paráfrase', "pergunta": 'Para que serve um filtro antispam?', "doc": 'cartilha', "esperado": ['7.9']},
    {"id": 71, "tipo": 'paráfrase', "pergunta": 'Como usar contas e senhas com segurança?', "doc": 'cartilha', "esperado": ['8.1']},
    {"id": 72, "tipo": 'paráfrase', "pergunta": 'Como elaborar uma senha segura?', "doc": 'cartilha', "esperado": ['8.2']},
    {"id": 73, "tipo": 'paráfrase', "pergunta": 'Como funciona o gerenciamento de contas e senhas?', "doc": 'cartilha', "esperado": ['8.4']},
    {"id": 74, "tipo": 'paráfrase', "pergunta": 'Qual a diferença entre criptografia de chave simétrica e assimétrica?', "doc": 'cartilha', "esperado": ['9.1']},
    {"id": 75, "tipo": 'paráfrase', "pergunta": 'O que é função de resumo (hash)?', "doc": 'cartilha', "esperado": ['9.2']},
    {"id": 76, "tipo": 'paráfrase', "pergunta": 'O que é assinatura digital?', "doc": 'cartilha', "esperado": ['9.3']},
    {"id": 77, "tipo": 'paráfrase', "pergunta": 'O que é certificado digital?', "doc": 'cartilha', "esperado": ['9.4']},
    {"id": 78, "tipo": 'paráfrase', "pergunta": 'Quais programas de criptografia posso usar?', "doc": 'cartilha', "esperado": ['9.5']},
    {"id": 79, "tipo": 'paráfrase', "pergunta": 'Quais cuidados tomar ao usar criptografia?', "doc": 'cartilha', "esperado": ['9.6']},
    {"id": 80, "tipo": 'paráfrase', "pergunta": 'Como ter segurança em conexões Web?', "doc": 'cartilha', "esperado": ['10.1']},
    {"id": 81, "tipo": 'paráfrase', "pergunta": 'Quais são os tipos de conexão?', "doc": 'cartilha', "esperado": ['10.1.1']},
    {"id": 82, "tipo": 'paráfrase', "pergunta": 'Como usar redes sociais com segurança?', "doc": 'cartilha', "esperado": ['11.1']},
    {"id": 83, "tipo": 'paráfrase', "pergunta": 'Como administrar contas de usuários no computador?', "doc": 'cartilha', "esperado": ['12.1']},
    {"id": 84, "tipo": 'paráfrase', "pergunta": 'Quais cuidados gerais devo ter com redes?', "doc": 'cartilha', "esperado": ['13.1']},
    {"id": 85, "tipo": 'paráfrase', "pergunta": 'Como funciona o Bluetooth e quais são os riscos?', "doc": 'cartilha', "esperado": ['13.3']},
    {"id": 86, "tipo": 'ruído', "pergunta": 'Bom dia! Pode me explicar o artigo 5?', "doc": 'Lei_geral', "esperado": ['5']},
    {"id": 87, "tipo": 'ruído', "pergunta": 'Oi, tudo bem? O que é cookie?', "doc": 'cartilha', "esperado": ['6.1']},
    {"id": 88, "tipo": 'ruído', "pergunta": 'Por favor, o que é a ANPD? Obrigado.', "doc": 'Lei_geral', "esperado": ['55']},
    {"id": 89, "tipo": 'ruído', "pergunta": 'Olá! Quero saber o que é um rootkit, obrigado.', "doc": 'cartilha', "esperado": ['4.7']},
    {"id": 90, "tipo": 'ruído', "pergunta": 'Poderia me dizer o que é criptografia, por favor?', "doc": 'cartilha', "esperado": ['9.1']},
    {"id": 91, "tipo": 'ruído', "pergunta": 'Eu gostaria de saber o que diz o art. 7, valeu.', "doc": 'Lei_geral', "esperado": ['7']},
    {"id": 92, "tipo": 'ruído', "pergunta": 'Oi, qual é a seção 12.3?', "doc": 'cartilha', "esperado": ['12.3']},
    {"id": 93, "tipo": 'ruído', "pergunta": 'Bom dia, me explica o que é spam?', "doc": 'cartilha', "esperado": ['5.1']},
    {"id": 94, "tipo": 'fora-do-escopo', "pergunta": 'Qual é a capital da França?', "doc": None, "esperado": []},
    {"id": 95, "tipo": 'fora-do-escopo', "pergunta": 'Como faço um bolo de chocolate?', "doc": None, "esperado": []},
    {"id": 96, "tipo": 'fora-do-escopo', "pergunta": 'Qual o horário de funcionamento da empresa?', "doc": None, "esperado": []},
    {"id": 97, "tipo": 'fora-do-escopo', "pergunta": 'Quanto custa um notebook?', "doc": None, "esperado": []},
    {"id": 98, "tipo": 'fora-do-escopo', "pergunta": 'Qual a previsão do tempo para amanhã?', "doc": None, "esperado": []},
    {"id": 99, "tipo": 'fora-do-escopo', "pergunta": 'Quem ganhou a última Copa do Mundo?', "doc": None, "esperado": []},
    {"id": 100, "tipo": 'multi-documento', "pergunta": 'Qual a relação entre incidente de segurança e a comunicação à ANPD?', "doc": 'Lei_geral', "esperado": ['48']},
]

resultados = []
for p in PERGUNTAS_TESTE:
    q = comprimir_pregunta(p['pergunta'])
    rec = recuperar(q, TOP_K)
    if p['tipo'] == 'fora-do-escopo':
        ok = len(rec) == 0
    else:
        ok = acertou(rec, p['doc'], p['esperado'])
    obtido = [f"{r['titulo'][:12]} {referencia_de(r) or r['seccion'][:20]}" for r in rec] or ['(nada)']
    resultados.append({'id': p['id'], 'tipo': p['tipo'], 'ok': ok, 'pergunta': p['pergunta'],
                       'esperado': ', '.join(p['esperado']) or 'nada', 'obtido': ' | '.join(obtido)})

df = pd.DataFrame(resultados)
total_ok, total = int(df['ok'].sum()), len(df)
print(f"Top-{TOP_K}: {total_ok}/{total} acertos ({100 * total_ok / total:.0f}%)\n")
print(df.groupby('tipo')['ok'].agg(['sum', 'count']).rename(columns={'sum': 'acertos', 'count': 'total'}))
falhas = df[~df['ok']]
if len(falhas):
    print('\nFalhas:')
    for _, f in falhas.iterrows():
        print(f"#{f['id']} {f['pergunta']}\n    esperado: {f['esperado']} | obtido: {f['obtido']}")